# Chapitre 11 - Cours 3 : bases de données et flux, la capture des changements (CDC)

Les courtiers à journal ont emprunté des idées aux bases de données. Ce notebook fait le
chemin inverse : appliquer aux bases de données des idées venues des flux. Le point de
départ est simple : **chaque écriture dans une base est un événement**, que l'on peut
capturer, stocker et traiter.

Il couvre le début de la section *Databases and Streams* : l'introduction, *Keeping
Systems in Sync*, *Change Data Capture* (*Implementing change data capture*, *Initial snapshot*,
*Log compaction*, *API support for change streams*). L'event sourcing et l'immutabilité
sont l'objet du cours 4.

**Plan.**

1. Le journal de réplication est déjà un flux.
2. Garder des systèmes en phase : les deux défauts de la double écriture.
3. La capture des changements (CDC), implémentée sur SQLite.
4. L'instantané initial et sa position dans le journal.
5. La compaction du journal.
6. Les interfaces de flux de changements.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import streams as st
from ddia.diagrams import Diagram
from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)

## 1. Le journal de réplication est déjà un flux (section *Databases and Streams*)

Un **journal de réplication** (chapitre 5) est un flux d'événements
d'écriture produit par le leader ; les suiveurs l'appliquent à leur copie et obtiennent
les mêmes données. C'est un cas particulier du principe de **réplication de machine à
états** (state machine replication, chapitre 9) : si chaque événement est
une écriture déterministe et que toutes les répliques traitent **les mêmes événements dans
le même ordre**, elles aboutissent au même état. Vérifions-le, et voyons ce qui se passe
quand l'ordre diffère : les opérations sur une même clé ne commutent pas.

In [2]:
ops = [("x", "set", 5), ("x", "add", 3), ("y", "set", 1), ("x", "mul", 2), ("y", "add", 4), ("x", "set", 0),
       ("x", "add", 7)]

def appliquer(etat, op):
    k, f, v = op
    etat = dict(etat)
    etat[k] = {"set": v, "add": etat.get(k, 0) + v, "mul": etat.get(k, 0) * v}[f]
    return etat

def rejouer(ordre):
    e = {}
    for op in ordre:
        e = appliquer(e, op)
    return e

r1, r2 = rejouer(ops), rejouer(ops)
melange = [ops[i] for i in rng.permutation(len(ops))]
print("réplique 1 :", r1, "| réplique 2 (même ordre) :", r2)
print("réplique 3 (ordre différent) :", rejouer(melange))
etats = {}
for s in range(200):
    o = [ops[i] for i in np.random.default_rng(s).permutation(len(ops))]
    etats[str(sorted(rejouer(o).items()))] = etats.get(str(sorted(rejouer(o).items())), 0) + 1
print(f"200 ordres aléatoires donnent {len(etats)} états finaux différents")

réplique 1 : {'x': 7, 'y': 5} | réplique 2 (même ordre) : {'x': 7, 'y': 5}
réplique 3 (ordre différent) : {'y': 5, 'x': 8}
200 ordres aléatoires donnent 34 états finaux différents


Le même journal dans le même ordre donne le même état ; changer l'ordre de ces sept
opérations donne 34 états finaux différents sur 200 ordres tirés au hasard. Toute la suite du cours en
découle : pour qu'un système dérivé reste fidèle à une base, il faut lui faire appliquer
**les mêmes écritures, dans l'ordre où la base les a appliquées**.

> **À retenir.** Une base de données est un état ; son journal de réplication est le flux
> d'écritures qui produit cet état. Mêmes événements, même ordre, traitement déterministe :
> même état.